In [38]:
from flask import Flask, jsonify, request, url_for, render_template
from flask_cors import CORS

In [67]:
from keras.models import load_model
import joblib
import numpy as np
import os

# Check if the vectorizer file exists
if not os.path.exists('tfidf_vectorizer.pkl'):
    raise FileNotFoundError("The TF-IDF vectorizer file was not found. Please make sure it exists.")

# Load the trained Keras model


def get_analysis(new_article):
    saved_model_nn = load_model('neural_network_model_keras.h5')

# Load the fitted TF-IDF vectorizer
    tfidf_vect = joblib.load('tfidf_vectorizer.pkl')

    # Print the input article to check if it's correct
    print(f"Input article: '{new_article}'")  # Debugging line to check the article

    # Ensure the new article is a valid string
    if not new_article:
        raise ValueError("The input article is empty or invalid.")

    new_article = [new_article]  # Make sure it's a list, as the vectorizer expects this

    # Transform the new article using the fitted TF-IDF vectorizer
    new_article_tfidf = tfidf_vect.fit_transform(new_article)

    expected_shape = (1, 42005)  # Model expects this shape

    if new_article_tfidf.shape[1] != expected_shape[1]:
        # Manually pad the feature vector with zeros if the length is shorter than expected
        padded_article_tfidf = np.pad(new_article_tfidf.toarray(), ((0, 0), (0, expected_shape[1] - new_article_tfidf.shape[1])), 'constant', constant_values=0)
        print(f"Shape after padding: {padded_article_tfidf.shape}")
    else:
        padded_article_tfidf = new_article_tfidf.toarray()

    # Predict using the neural network model
    probabilities_nn = saved_model_nn.predict(padded_article_tfidf)
    
    return probabilities_nn[0][0]



In [68]:
app = Flask(__name__) 
CORS(app) 

@app.route("/")
def index():
    return render_template("index.html")

current_news = ""

@app.route("/data", methods=["GET", "POST"]) 
def dataex():
    global current_news
    if request.method == "POST": 
        data = request.get_json()
        #print(data, flush = True)
        current_news = data["news"]
        return {"message": "Success!"}
    
    elif request.method == "GET":
        result = get_analysis(str(current_news))
        return jsonify({"result": result.tolist()})
    
if __name__ == '__main__':
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:03] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:03] "GET /static/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:03] "GET /static/icons8loader.gif HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:03] "GET /static/dynamic.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:05] "POST /data HTTP/1.1" 200 -


Input article: 'indians landed on mars'
Shape after padding: (1, 42005)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step


INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:06] "GET /data HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:20] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:20] "GET /static/icons8loader.gif HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:20] "GET /static/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:20] "GET /static/dynamic.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:21] "POST /data HTTP/1.1" 200 -


Input article: 'kjdhfwff fdefedf'
Shape after padding: (1, 42005)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step


INFO:werkzeug:127.0.0.1 - - [21/Dec/2024 04:53:22] "GET /data HTTP/1.1" 200 -
